# ថ្នាក់ទី ១៣ - ចងចាំភ្នាក់ងារជាមួយក្រាហ្វិកចំណេះដឹង Cognee


## ការរៀបចំ

កំណត់ត្រានេះបង្ហាញពីវិធីសាស្ត្របង្កើត **ជំនួយការកូដ** មានបញ្ញាឥស្សរិយយស ជាមួយនឹងចងចាំដាច់ដោយឡែកដោយប្រើបែបបទចំណេះដឹង [**Cognee**](https://www.cognee.ai/) និង **Microsoft Agent Framework** (MAF)។

Cognee បម្លែងអត្ថបទដែលមិនមានរចនាសម្ព័ន្ធទៅជាប្រាក់បញ្ជាក់ចំណេះដឹងដែលមានរចនាសម្ព័ន្ធ និងអាចសួរសំណួរ ត្រូវបានគាំទ្រដោយការតមមូលវ៉ិចទ័រ — ផ្តល់ចំណាំអតិភាព និងមានទំនាក់ទំនងប្រកបដោយអារម្មណ៍ឲ្យភ្នាក់ងាររបស់អ្នក។

### អ្វីដែលអ្នកនឹងរៀន
1. **បង្កើតតារាងចំណេះដឹង** ៖ បម្លែង​ប្រវត្តរូបអ្នកអភិវឌ្ឍន៍ និងអនុវត្តល្អៗទៅជាចំណេះដឹងដែលមានរចនាសម្ព័ន្ធ និងអាចសួរសំណួរបាន។
2. **រួមបញ្ចូល Cognee ជាមួយ MAF** ៖ ប្រើមុខងារ `@tool` ដើម្បីឲ្យភ្នាក់ងារ MAF អាចសួរចំណេះដឹងពីតារាង Cognee។
3. **ការជជែកសន្ទនាដែលយល់ពីសម័យសមិទ្ធ** ៖ រក្សាកប្បាសមិនផ្លាស់ប្ដូរពីសំណួរច្រើននៅក្នុងសម័យសមិទ្ធដូចគ្នា។
4. **ចងចាំរយៈពេលវែង** ៖ រក្សាទុកចំណេះដឹងសំខាន់ៗឲ្យឆាប់ស្វែងរកវិញនៅក្នុងការជជែកថ្មីៗ។

### លក្ខខណ្ឌមុន
- Python 3.9+
- Redis រត់នៅក្នុងเครื่อง (`docker run -d -p 6379:6379 redis`) សម្រាប់ការគ្រប់គ្រងសម័យសមិទ្ធ
- កូនសោ API LLM (ឧ. OpenAI) — កំណត់ `LLM_API_KEY` ក្នុង `.env`
- `CACHING=true` ក្នុង `.env` ( តម្រូវសម្រាប់សម័យ Cognee )
- គម្រោង Microsoft Foundry ជាមួយម៉ូដែលជជែកបានបង្ហោះ
- `AZURE_AI_PROJECT_ENDPOINT` និង `AZURE_AI_MODEL_DEPLOYMENT_NAME` ក្នុង `.env`
- បានAuthenticate Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## ប្រភេទអង្គចងចាំរបស់ភ្នាក់ងារ

សៀវភៅកំណត់ត្រានេះស្វែងយល់ពីប្រភេទអង្គចងចាំទាំងបីដូចក្នុងសៀវភៅកំណត់ត្រាមេរៀនទី ១៣ ប៉ុន្តេញាក់ Cognee ជាបណ្តារចងចាំរយៈពេលវែង៖

| ប្រភេទអង្គចងចាំ | មេកានិច | រយៈពេលជីវិត |
|---|---|---|
| **កំពុងដំណើរការ** | `agent.create_session()` (MAF) | សន្ទនា​ផ្ទាល់​តែ​មួយ |
| **រយៈពេលខ្លី** | កេសស្យុងរក្សារសម័យ Cognee (Redis) | កេសស្យុង​តែ​មួយ |
| **រយៈពេលវែង** | គំនូសចំណេះដឹង Cognee + វ៉ិចទ័រ | ធម្មតា |

### វិស្ថានអង្គចងចាំរបស់ Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## រៀបចំផ្ទុក Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## ផ្នែក ១ — ការបង្កើតមូលដ្ឋានចំណេះដឹង

យើងបញ្ចូលទិន្នន័យចំនួនបីប្រភេទ ដើម្បីបង្កើតមូលដ្ឋានចំណេះដឹងដ៍ទូលំទូលាយសម្រាប់ជំនួយការកូដរបស់យើង៖

1. **ប្រវត្តិរូបអ្នកអភិវឌ្ឍន៍** — ជំនាញផ្ទាល់ខ្លួន និងប្រវត្តិផ្នែកបច្ចេកទេស
2. **ការ​អនុវត្ត​អាស្រ័យ​លើ Python ល្អបំផុត** — សមាគម Zen របស់ Python ជាមួយមគ្គុទេសក៍អនុវត្តន៍
3. **ការពិភាក្សាប្រវត្តិសាស្ត្រ** — ការសួរសំនួរនិងចម្លើយមុនៗ រវាងអ្នកអភិវឌ្ឍន៍ និងជំនួយការប្រព័ន្ធ AI


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## បង្ហាញ​ការ​វិស្វកម្ម​ជាក់ស្តែង​នៃ​ជំនាញ

Cognee អាចបង្ហាញការវិស្វកម្ម HTML ដែលអាចអន្តរាគមន៍របស់អង្គធាតុ និងទំនាក់ទំនងដែលវាបានដកយក។ 


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## ជួញដូរចងចាំជាមួយ Memify

`memify()` វិភាគក្របខ័ណ្ឌចំណេះដឹង ហើយបង្កើតច្បាប់ឆ្លាតវៃ — ដើម្បីស្គាល់គំរូ ការអនុវត្តល្អបំផុត និងទំនាក់ទំនងរវាងគំនិត។


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## ផ្នែក 2 — ភ្នាក់ងារ MAF ជាមួយឧបករណ៍ Cognee

ឥឡូវនេះយើងបង្កើតភ្នាក់ងារ MAF ដែលអាចស្នក្សាសំណួរចំពោះគ្រាប់ដំណោះស្រាយតាមអក្សរសាស្ត្រ Cognee តាមរយៈមុខងារ `@tool`។ វាអនុញ្ញាតឲ្យភ្នាក់ងារប្រើអំណាចពេញលេញនៃការសិន្សាថ្មីដែលមានចំណេះដឹងអំពីក្រាហ្វិច ខណៈពេលរក្សាសេចក្តីផ្ទាល់ខ្លួនក្នុងការសន្ទនាតាមផ្ទាំងជួបប្រជុំ។


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## ការចងចាំការងារជាមួយសម័យកាល

`AgentSession` (ដែលបានបង្កើតតាមរយៈ `agent.create_session()`) ផ្តល់នូវការចងចាំការងារផ្នែកក្នុងសម័យកាល។ អ្នកភ្នាក់ងារ​អាចយោងត្រឡប់ទៅសារ​មុនៗ ខណៈពេលដែលពួកគេសួរសុខទុក្ខទៅកាន់ក្រាហ្វ់ចំណេះដឹងរយៈពេលវែងរបស់ Cognee ។


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## សម័យថ្មី — សេចក្តីចងចាំរយៈពេលវែងនៅតែបន្ត

ការចាប់ផ្ដើមសម័យថ្មីមួយនឹងលាងចោលចងចាំកំពុងដំណើរការ ប៉ុន្តែក្រាហ្វនៃចំណេះដឹង Cognee ក៏នៅតែអាចប្រើបាន។ អ្នកបញ្ជាទទួលអាចយកចំណេះដឹងរយៈពេលវែងដូចគ្នា ក្នុងការពិភាក្សាថ្មីទាំងស្រុងមួយបាន។


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## សង្ខេប

នៅក្នុងសៀវភៅកំណត់ត្រានេះ អ្នកបានបង្កើតជំនួយការកូដដែលបញ្ចូលគ្នារវាង **អនុស្សារមិ្ធការងាររបស់ MAF** (`agent.create_session()`) និង **គ្រោងចំណេះដឹងរយៈពេលវែងរបស់ Cognee**។

### អ្វីដែលអ្នកបានរៀន
1. **ការបង្កើតគ្រោងចំណេះដឹង**: Cognee គ្រប់គ្រងអត្ថបទដែលមិនមានរចនាសម្ព័ន្ធ ហើយបង្កើតគ្រោង + អនុស្សារមិ្ធវ៉ិចទ័រ។
2. **ការបន្ថែមតម្លៃលើគ្រោងជាមួយ memify**: ព័ត៌មានដែលបានដាក់ទៅជាការពិត និងទំនាក់ទំនងប្រសើរឡើងលើគ្រោងរបស់អ្នក។
3. **ការរួមបញ្ចូល MAF + Cognee**: មុខងារ `@tool` អនុញ្ញាតឲ្យភ្នាក់ងារ MAF ស្ទាប់សំណួរជាមួយគ្រោង Cognee ដោយធម្មជាតិ។
4. **អនុស្សារមិ្ធការងារ + អនុស្សារមិ្ធរយៈពេលវែង**: `AgentSession` (តាមរយៈ `agent.create_session()`) ផ្តល់បរិបទសម័យសម័យ ខណៈពេល Cognee ផ្តល់ចំណេះដឹងប្រចាំកាល។
5. **ការស្វែងរកតាមតម្រងជាមួយ NodeSets**: មានគោលដៅជាប្រភេទរងនៃគ្រោងចំណេះដឹង (ឧ. គោលការណ៍តែមួយ)។

### ចំនុចសំខាន់
- **Cognee** បំលែងអត្ថបទដើមទៅជាអនុស្សារមិ្ធដែលមានរចនាសម្ព័ន្ធ និងមានការយល់ដឹងអំពីទំនាក់ទំនង — មានអំណាចជាងហាងវ៉ិចទ័រ។  
- **មុខងារ `@tool`** ជាស្ពាន់ភ្ជាប់ភ្នាក់ងារ MAF និងប្រព័ន្ធចំណេះដឹងក្រៅយ៉ាងច្បាស់លាស់។  
- **`AgentSession`** (តាមរយៈ `agent.create_session()`) រក្សាបរិបទសម្ភាសន៍ប្លែកពីចំណេះដឹងប្រចាំកាល។  
- គ្រោងចំណេះដឹងដូចគ្នាសម្រាប់សម័យ និងភ្នាក់ងារច្រើន។  

### ការអនុវត្តនៅក្នុងពិភពលោកពិត
- **ជំនួយការអភិវឌ្ឍន៍**: ពិនិត្យកូដ វិភាគករណីកើតឡើង ជំនួយការរចនាស្ថាបត្យកម្ម  
- **ជំនួយការចំពោះអតិថិជន**: ជំនួយភ្នាក់ងារក្នុងឯកសារផលិតផល FAQ និងកំណត់ត្រា CRM  
- **ជំនួយការខាងជំនាញផ្ទៃក្នុង**: ជំនួយការច្បាប់ នីតិវិធី ឬសន្តិសុខ ដោយយោងទៅលើមគ្គុទេសក៍  
- **ស្រទាប់ទិន្នន័យរួម**: បញ្ចូលទិន្នន័យរចនាសម្ព័ន្ធ និងមិនរចនាសម្ព័ន្ធទៅក្នុងគ្រោងដែលអាចស្វែងរកបាន  

### ជំហានបន្ទាប់
- សាកល្បងការយល់ដឹងពេលវេលានៅក្នុង Cognee  
- កំណត់តម្លៃ OWL សម្រាប់គុណភាពគ្រោងតាមវិស័យ  
- បន្ថែមមូលដ្ឋានមតិអ្នកប្រើដើម្បីបង្កើនការទាញយកឲ្យប្រសើរឡើងតាមពេលវេលា  
- ពង្រីកទៅប្រព័ន្ធភ្នាក់ងារច្រើនដែលចែករំលែកស្រទាប់អនុស្សារមិ្ធ Cognee ដូចគ្នា  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
